# Семантический поиск по схемам чартов

**Этот блокнот — единственный source of truth по проекту.** Здесь зафиксирована постановка, все принятые решения и план выполнения. Любой запуск работы начинается с чтения этого ноутбука.

## Текущий статус

- ✅ Постановка зафиксирована
- ✅ План экспериментов утверждён
- ✅ Корпус загружен и отфильтрован: `data/recipes.json` (67 рецептов, 4 датасета). Оригинал — [chartRecipes.json](chartRecipes.json), не модифицировался.
- ✅ **Шаг 1: роли полей размечены** → `data/field_roles.json` (108 полей: 12 temporal / 45 categorical / 46 metric / 5 entity / 0 text).
- ✅ **Шаг 2: Generator-pass** → `data/eval/queries_raw.jsonl` (334 запроса).
- ✅ **Шаг 3: Judge-pass** → `data/eval/queries.jsonl` (334 записи). Avg `|template|` = 4.51, max = 7.
- ✅ **Шаг 4: Ручной ревью пройден.**
- ✅ **Шаг 5: Phase 1 — основной грид (30 прогонов).** Победитель `(M*, S*)` = `S4 + Qodo-Embed-1-1.5B`. macro `recall@5_template` = 0.742 (структурный охват), `recall@5_strict` = 0.956 (Hit@5 source).
- ✅ **Шаг 6: Phase 2 — routing.** Лучший роутер ≈ `mix(4,1)` или `cascade τ=0.85` (macro 0.412–0.414). Прирост от routing ~1%. Сценарий C почти не нужен на нашем eval — но это **может быть артефактом перекоса eval-сета** (см. ниже).
- ✅ **Шаг 7: Phase 1.5 — held-out leave-one-out (production-realistic выбор стратегии).** Все 30 пар (M, S) прогнаны с leave-one-out source. Победитель **остался `S4 + Qodo-Embed-1-1.5B`** (held-out macro = **0.642** vs Phase 1: 0.742, Δ = −0.100). Главное узкое место — taxi (0.239) и T6 cross-dataset (0.150) — это цели Phase 3.
- ✅ **Шаг 8: Phase 3 — BM25 + Hybrid (RRF)** на победителе. **Hybrid (BM25 + dense) — рекомендован для прода**: на in-dataset паритет (0.640 vs 0.642 dense), на T6 cross-dataset **major win** — T6_strict 0.474 vs 0.263 dense (почти 2×). На taxi (in-dataset дыра) gain малый: BM25 alone 0.254, Hybrid 0.228 vs dense 0.239 — taxi-плато не пробивается. Cross-encoder reranker оставлен на опциональный этап.
- ✅ **Шаг 10: Итоговый отчёт** — финальная markdown-ячейка в конце ноутбука. Сводная таблица всех Phase, рекомендация для прода (Hybrid BM25+dense), путь к 0.8, что не получилось, anti-pattern'ы.
- ⬜ Phase 4 (fine-tuning) — опционально, не рекомендуется без расширения eval-сета.

### Снапшот корпуса

| Параметр | Значение |
|---|---|
| Рецептов | 67 |
| Датасетов | 4 |
| Распределение | `33h8c3n5nbien`: 29, `b60rhj4luj0y3`: 19, `pzf0mu9kgqz4k`: 12, `omhpbh1k83ao8`: 7 |
| Multi-layer (2 layers) | 4 рецепта (6%) |
| Description (median / mean) | 39 / 64 символов |

Удалены: 16 рецептов с пустым `datasetId`, 3 рецепта `rmcxc5tz38foc`, 1 рецепт `w8nnpr3kdsyko`. Каждому рецепту присвоен стабильный `recipe_id` вида `r_0000`. Датасет `omhpbh1k83ao8` (7 рецептов) принят со known шумом in-dataset метрик — оставлен сознательно.

### Снапшот eval-сета

| Тип | Кол-во | avg `\|strict\|` | avg `\|template\|` |
|---|---|---|---|
| T1 (близкая перефразировка) | 67 | 1.15 | 4.88 |
| T2 (структурный намёк) | 67 | 1.15 | 4.88 |
| T3 (сущности/числа) | 67 | 1.10 | 4.88 |
| T4 (косвенная формулировка) | 67 | 1.15 | 4.88 |
| T5 (hard negative «близнец») | 19 | 0.00 | 5.63 |
| T6 (cross-dataset trigger) | 19 | 1.00 | 4.74 |
| T7 (no-match) | 28 | 0.00 | 0.00 |
| **всего** | **334** | **0.97** | **4.51** |

`fallback_label` распределение: in_dataset 268, cross_dataset 35, no_match 31.

`relevant_template` — целевые ~5 релевантных схем на запрос (агент использует top-k retrieve как образцы → метрика recall@5 имеет реальное разрешение, max=7 → recall@5 достижим до 5/7 = 71%). `relevant_strict` — узкое подмножество для семантически точного матча.

### Откуда берутся эти ~5 релевантных шаблонов

Из 4.92 шаблонов (на T1–T4 в среднем): **~3.15 in-dataset + ~1.73 из других датасетов** (≈ 65/35). Но distribution **сильно варьируется** по датасету запроса:

| `current_dataset` | Запросов | avg total | avg in-ds | avg cross | Доминирует |
|---|---|---|---|---|---|
| `pzf0mu9kgqz4k` (observ) | 59 | 6.02 | **4.46** | 1.56 | in-ds (74%) |
| `33h8c3n5nbien` (taxi) | 125 | 5.07 | **3.30** | 1.77 | in-ds (65%) |
| `b60rhj4luj0y3` (retail) | 82 | 3.39 | 1.99 | 1.40 | mixed |
| `omhpbh1k83ao8` (films) | 40 | 5.95 | 1.77 | **4.17** | cross-ds (70%) |

Films (всего 7 рецептов) тянет шаблоны в основном из других датасетов — это **подтверждает необходимость fallback** (сценарий C). Observability наоборот, имеет много структурно похожих в одном датасете — fallback там почти не нужен. На retrieval-стороне: для in-dataset метрики каждый датасет имеет от 1.77 до 4.46 релевантных в среднем — recall@5 max везде ≈ 100%.

### Замечания по корпусу

- **Датасет `b60rhj4luj0y3`** (19 рецептов) почти целиком состоит из демо/template-рецептов с генерическими описаниями: `«Один показатель»`, `«С группировкой по измерению»`, `«Размер L (Wizard)»`, `«таблица-2»`. Описания не несут семантики — для T1/T4 запросы формулировались через структуру recipe (поля, фильтры).
- **Датасет `pzf0mu9kgqz4k`** (12 рецептов) — мониторинг/observability, описания на английском короткие (`«errors by action»`, `«Int Prod Median»`).
- **Датасет `33h8c3n5nbien`** (29 рецептов) — такси, описания содержательные на русском, эмодзи в именах полей.
- **Датасет `omhpbh1k83ao8`** (7 рецептов) — фильмы, длинные содержательные описания.

См. раздел «Что делать дальше» в конце ноутбука.

---

## Задача

Пользователь делает запрос на построение чарта на естественном языке. LLM-агент собирает итоговую схему **не с нуля**, а на основе нескольких наиболее подходящих **уже существующих рецептов** из корпуса.

Цель проекта — построить и сравнить **сервис retrieval-а**, который по запросу возвращает top-k рецептов. Это серия экспериментов с метриками, по итогам которой выбирается лучший подход.

Формат рецепта (см. [chartRecipes.json](chartRecipes.json)):

```json
{
  "description": "Топ комедийных сериалов 1990х",
  "recipe": {
    "datasetId": "omhpbh1k83ao8",
    "layers": [{
      "type": "line | column | bar | area | donut | pie | flatTable | pivotTable | metric | column100p | bar100p | area100p",
      "x": [{"title": "..."}],
      "y": [{"title": "..."}],
      "y2": [...], "colors": [...], "measures": [...], "columns": [...],
      "filters": [{"field": {"title": "..."}, "operation": "IN|GT|LT|...", "values": [...]}],
      "sorting": [{"title": "...", "direction": "ASC|DESC"}]
    }]
  }
}
```

## Подход

Векторизация рецептов и запроса + поиск по косинусному сходству. **Сценарий C (гибрид):** сначала ищем в датасете пользователя, при необходимости fallback на весь корпус как поиск **структурных шаблонов** (рецепт из другого датасета используется агентом как форма чарта, а агент адаптирует имена полей).


## Определение релевантности (мульти-меточная, два уровня)

Для пары `(query, recipe)` judge ставит **две независимые булевые метки**:

| Метка | Условие | Используется для |
|---|---|---|
| `relevant_strict` | (1) семантический матч: `description` рецепта отвечает на запрос; **И** (2) типы чартов эквивалентны; **И** (3) роли полей совпадают (с фази-толерантностью) | метрика in-dataset (узкая) |
| `relevant_template` | только условия (2) и (3) — без семантики, **с cap K=7** наиболее похожих | главная метрика, ~5 релевантных на запрос |

`strict ⊆ template`. Датасет — поле метаданных, не входит в определение релевантности; метрики разрезаются по нему.

### (2) Эквивалентность типов чарта

Группы эквивалентности (после анализа корпуса):

- `{column, bar, column100p, bar100p}` — бары (включая stacked-100%)
- `{line, area, area100p}` — линии/области (включая area-100%)
- `{donut, pie}` — круговые
- `{metric}` — single value (≡ `kpi` из плана; в корпусе только `metric`)
- `flatTable` — отдельно
- `pivotTable` — отдельно

`scatter` и `treemap` отсутствуют в отфильтрованном корпусе.

### (3) Фази-совпадение ролей полей в позициях

> **Изменение vs первоначальная спека (зафиксировано 2026-05-10):** перешли от строгих equivalence-классов к **попарной fuzzy-совместимости + cap по similarity** — для большинства запросов нужно ~5 кандидатов в template, и строгая спека давала много singleton-кластеров и избыточные кластеры размера 14+.

Каждому полю на этапе препроцессинга присваивается **роль** из таксономии:

| Роль | Примеры |
|---|---|
| `temporal` | Год выпуска, Дата заказа, Месяц, Квартал |
| `categorical` | Тип, Жанр, Страна, Регион |
| `metric` | Рейтинг, Сумма, Число фильмов, Число голосов |
| `entity` | Ссылка, Название фильма, ID |
| `text` | свободный длинный текст |

> **Note:** роль `metric` (числовое измерение) и тип чарта `metric` (single-value display) — омонимы. В коде использовать разные имена: `FieldRole.METRIC` vs `ChartType.METRIC`.

**Правила базовой совместимости** (фильтры в матчинг НЕ входят):

| Тип | Что должно совпасть |
|---|---|
| line / area / column / bar (включая `*100p`) | **set** уникальных ролей в `x` совпадает **И** **set** уникальных ролей в `y`. Color и количество полей игнорируются. |
| donut / pie | set ролей в `measures` |
| flatTable | мультимножество ролей в `columns` различается на **≤2 роли** (релаксация: спека говорила ≤1, но на корпусе 67 это давало много singletons). |
| pivotTable | объединённое мультимножество (rows ∪ columns ∪ measures) различается на **≤2 роли** |
| metric | set ролей в `measures` |

**Cap K=7 по similarity.** Базовая совместимость даёт компат-сеты до 14+. Чтобы recall@5 имел разрешение (без cap caps на 5/14=36%), для каждого рецепта берём top-7 по similarity score:

| Компонент score | Балл |
|---|---|
| Точный тип чарта (column == column, не column ≈ bar) | +2 |
| Совпадение количества полей в позициях (`x`, `y`, `colors`, `measures`, ...) | +0.5 за каждую |
| Разница числа фильтров ≤2 | +1 |
| Тот же `datasetId` | +0.5 |

Source-of-self всегда score = max → попадает в top-7 первым.

**Свойства:**
- Симметричность базовой совместимости — да.
- Симметричность ПОСЛЕ cap — нет (A может оставить B в top-7, а B предпочесть других). Это OK для retrieval-задачи: каждый запрос смотрит на свой собственный source.
- Не транзитивность — да (для table'ов с ≤2). Представление кластерами невозможно, используется попарный список (`data/recipe_template_candidates.json`).

### (4) Multi-layer рецепты

4 рецепта (6%) имеют 2 layers. Правила:

- **Тип чарта = отсортированный кортеж** типов layers, например `("column", "line")`. Эквивалентность типов внутри кортежа применяется поэлементно (после сортировки).
- **Совпадение ролей** — каждый layer проверяется независимо по правилам своего типа. Layers сопоставляются по позиции после сортировки по типу.
- **Сериализация** — все layers последовательно: разделитель ` | ` для F2 (`flat_kv`), `Дополнительный слой: ...` для S4 (`linearize_to_text`).

## Eval-сет

Eval-сет собирается в **два прохода**:

**1. Generator** — для каждого рецепта генерируется ~5–7 запросов, по одному на тип:

| Тип | Что проверяет | Пример (для рецепта «комедийные сериалы 90х») |
|---|---|---|
| T1 | близкая перефразировка | «лучшие комедии-сериалы девяностых» |
| T2 | структурный намёк | «топ-100 списком: комедии-сериалы 90-х» |
| T3 | сущности/числа | «комедии 1995–1999, рейтинг» |
| T4 | косвенная формулировка | «что смотреть из старых смешных сериалов» |
| T5 | hard negative «близнец» | «комедийные **фильмы** 90х» (если есть рецепт про фильмы) |
| T6 | cross-dataset trigger | тот же intent, но в текущем датасете нет такого рецепта |
| T7 | no-match | «прогноз погоды в Москве» |

**Принципы генерации:**
- T1 ≠ копия description дословно (иначе любой dense тривиально матчит); должен передавать смысл другими словами.
- T2 содержит подсказку о форме: «линией», «столбиками», «списком», «топом», «процентами», «круговой».
- T3 опирается на конкретные значения из `filters` recipe (годы, категории, страны).
- T4 — разговорный/общий вопрос без слов из description.
- T5 генерируется только при наличии «близнеца» в корпусе (рецепт с похожей семантикой, но другой структурой/категорией).
- T6 и T7 генерируются **отдельным проходом** после T1–T4: T6 — для рецептов, имеющих структурный аналог в другом датасете; T7 — батч случайных не-релевантных запросов.

Для рецептов с генерическими описаниями (`«Один показатель»`, `«Размер L (Wizard)»`) T1/T4 опираются на структуру recipe (тип чарта, поля), не на description.

Целевой объём: **~400 запросов** на корпус 67 рецептов (≈5–6 запросов в среднем на рецепт + батч T7).

**2. Judge** — для каждого запроса видит запрос, его `source_recipe_id`, кандидатов (полное кросс-произведение или ограниченный set с hard negatives), роли полей. Возвращает обе булевые метки `relevant_strict` и `relevant_template`.

**Структура одной eval-записи:**

```json
{
  "query_id": "q_0001",
  "query": "как менялся рейтинг сериалов по годам",
  "current_dataset_id": "omhpbh1k83ao8",
  "query_type": "T4",
  "source_recipe_id": "r_002",
  "fallback_label": "in_dataset | cross_dataset | no_match",
  "relevant_strict_recipe_ids": ["r_002", "r_006"],
  "relevant_template_recipe_ids": ["r_002", "r_006", "r_011"],
  "judge_confidence": 0.85
}
```

**Ручной ревью:** случайная выборка 10–15% (~50–60 запросов) проверяется руками — оценка шума разметки. Если согласие <80% — итерируем judge-промпт.

## Метрика

**Recall@5** считается **per-dataset** и агрегируется в macro-average. Порядок внутри top-k не учитываем (downstream-агент обрабатывает весь набор как образцы):

$$\mathrm{recall@5}^{(q)} = \frac{|\mathrm{relevant}^{(q)} \cap \mathrm{top5}^{(q)}|}{|\mathrm{relevant}^{(q)}|}$$

### Зачем per-dataset разрез (важно)

Описания в датасетах **принципиально разные** по качеству:

| Датасет | Описания | S1 (только description) ожидаемо |
|---|---|---|
| `omhpbh1k83ao8` (films) | Длинные осмысленные RU | Сильно |
| `33h8c3n5nbien` (taxi) | Содержательные RU + эмодзи | Хорошо |
| `pzf0mu9kgqz4k` (observ) | Короткие EN (`errors by action`) | Средне |
| `b60rhj4luj0y3` (retail) | Демо (`«Один показатель»`) | Плохо |

Без разреза по датасетам метрика усреднится, и мы **не увидим главного**: какая стратегия вытаскивает retrieval, когда описания слабые. Это и есть основная гипотеза проекта — что схема в эмбеддинге компенсирует слабые описания.

### Macro vs micro

- **Micro-average** (средневзвешенное по числу запросов) — taxi (132) и retail (89) доминируют.
- **Macro-average** (среднее средних по датасетам) — каждый датасет получает равный вес.

**Headline-метрика — macro-average recall@5_template.** Tie-break: минимум по датасетам (consistency — никакой датасет не падает катастрофически). Дополнительно отчётно: per-dataset recall@5 во всех 4 датасетах + micro для сравнения.

### Метрики, которые считаем

- **`recall@5_template_macro`** — главная headline. Среднее средних `recall@5_template_in_dataset` по 4 датасетам.
- **`recall@5_template_micro`** — средневзвешенное (для сравнения).
- **`recall@5_template_per_dataset`** — таблица из 4 значений.
- **`recall@5_template_cross_dataset`** — на T6 запросах (relevant_template в других датасетах).
- **`recall@5_strict_*`** — то же на узких метках strict (вспомогательное; на T1–T4 strict ≈ {source}, метрика близка к Hit@5).
- `recall@1`, `recall@10` — для разрезов и анализа.
- `latency_ms_p50` — медианная латентность retrieve-вызова.
- `routing_accuracy` (только Phase 2) — для каскада (a): доля запросов с правильным решением о fallback.

**Замечание о CI:** на films (28 in-dataset queries) per-dataset recall@5 имеет 95% CI ±18%. Это явно отмечать в отчёте — films-разрез шумный.

## Phase 1 — основной грид: 10 стратегий × 3 модели = 30 прогонов

### Стратегии векторизации

| ID | Что эмбеддится |
|---|---|
| `S1` | `description` |
| `S2-F1` | `description + " " + json.dumps(recipe, ensure_ascii=False)` |
| `S2-F2` | `description + " " + flat_kv(recipe)` |
| `S3-F1@α` (α=0.25, 0.5, 0.75) | `α · v(description) + (1-α) · v(json.dumps(recipe))` |
| `S3-F2@α` (α=0.25, 0.5, 0.75) | `α · v(description) + (1-α) · v(flat_kv(recipe))` |
| `S4` | `description + " " + linearize_to_text(recipe)` |

**`flat_kv(recipe)` — flat key=value сериализация (для F2):**

Пример вывода:
```
type: line; x: Год выпуска; y: Рейтинг; colors: Тип; filters: Тип IN [Сериал]
```

Правила: `{type}; x: {x.title}; y: {y.title}; colors: {colors.title}; filters: {field.title} {op} [{values, joined by ', '}]`. Пустые поля пропускаем. Несколько фильтров — через `;`.

**`linearize_to_text(recipe)` — детерминированный NL-шаблон (для S4):**

Для line/column/area/bar:
```
Чарт типа «{type}». По оси X — поле «{x.title}». По оси Y — поле «{y.title}»{, и поле «{y2.title}» по второй оси Y}. Цветовая разбивка по полю «{colors.title}», если задана. Фильтры: {field.title} {operation} {values}, через «и».
```

Для donut/pie: `Круговая диаграмма по измерению «{measures.title}». Фильтры: …`

Для flatTable: `Таблица с колонками: {columns.title через запятую}. Сортировка: {sorting.title} {direction}. Фильтры: …`

После первого прогона можно подкрутить шаблон, если что-то очевидно теряется.

### Модели

| ID | Описание |
|---|---|
| `intfloat/multilingual-e5-large` | strong multilingual baseline, BERT-style. Использует префиксы `query:` / `passage:`. |
| `BAAI/bge-m3` | multilingual, dense + sparse + multi-vector (используем dense ветку) |
| `qodo-embed-1.5` | генеративная модель, адаптированная под эмбеддинги — другой архитектурный класс vs BERT |

### Зачем F1 + F2 одновременно

F1 (JSON) благоприятен для qodo (code-native), F2 (flat NL-friendly) — для e5/BGE. Главный вопрос Phase 1: важнее ли «правильный формат под модель» или «лучшая модель» — увидим, есть ли interaction `model × format`.

### Где запускаются эксперименты

**Все 30 прогонов оркестрируются из ячеек этого ноутбука** (раздел «Прогон Phase 1» ниже). Логика — в `src/`, ноутбук импортирует и вызывает. Сводная таблица результатов рендерится в этом же ноутбуке.

### Победитель

По итогам Phase 1 фиксируем `(M*, S*)` как лучшую пару по:

1. **Главное:** `recall@5_template_macro` (среднее средних по 4 датасетам — каждый датасет с равным весом).
2. **Tie-break:** минимум `recall@5_template_in_dataset` по датасетам (consistency: никакой датасет не падает катастрофически).
3. **Дополнительно к отчёту:** per-dataset разрез + `recall@5_template_cross_dataset` (на T6) + recall@1/recall@10 + latency.

## Прогон Phase 1

Все 30 экспериментов оркестрируются из ячеек ниже. Логика — в `src/`:
- `src/config.py` — список моделей/стратегий, пути.
- `src/indexers/serialize.py` — `flat_kv()` / `linearize_to_text()` / `json_dump()`.
- `src/indexers/embed.py` — обёртки над transformers, mean/CLS/last-token pooling, кеш на диске.
- `src/retrievers/dense.py` — cosine top-k с фильтрами.
- `src/eval/metrics.py` — recall@k, macro/micro, per-dataset, cross-dataset.
- `src/phase1.py` — runner полного грида.

Структура: setup → runner → результаты + выводы. Запуск idempotent (skip уже посчитанные).

In [1]:
# --- Setup: импорты и общие константы из src/ ---
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    MODELS, STRATEGIES, DATASETS,
    RECIPES_PATH, QUERIES_PATH, RESULTS_DIR, CACHE_DIR,
)

print(f"Models     ({len(MODELS)}): {[m.split('/')[-1] for m in MODELS]}")
print(f"Strategies ({len(STRATEGIES)}): {STRATEGIES}")
print(f"Total runs: {len(MODELS) * len(STRATEGIES)}")
print()
print(f"Recipes file:   {RECIPES_PATH.exists()}  ({RECIPES_PATH.relative_to(PROJECT_ROOT)})")
print(f"Queries file:   {QUERIES_PATH.exists()}  ({QUERIES_PATH.relative_to(PROJECT_ROOT)})")
print(f"Datasets:       {DATASETS}")

Models     (3): ['multilingual-e5-large', 'bge-m3', 'Qodo-Embed-1-1.5B']
Strategies (10): ['S1', 'S2-F1', 'S2-F2', 'S3-F1@0.25', 'S3-F1@0.5', 'S3-F1@0.75', 'S3-F2@0.25', 'S3-F2@0.5', 'S3-F2@0.75', 'S4']
Total runs: 30

Recipes file:   True  (data/recipes.json)
Queries file:   True  (data/eval/queries.jsonl)
Datasets:       ['33h8c3n5nbien', 'b60rhj4luj0y3', 'pzf0mu9kgqz4k', 'omhpbh1k83ao8']


In [2]:
# --- Phase 1 runner: 10 стратегий × 3 модели = 30 прогонов ---
# Логика в src/phase1.py. Idempotent: пропускает уже посчитанные.
# При первом запуске для каждой модели подкачиваются веса с HuggingFace
# (e5-large ~2.2GB, bge-m3 ~2.3GB, qodo-1.5B ~6GB).

import sys
sys.path.insert(0, str(PROJECT_ROOT))
from src.phase1 import run_phase1

written = run_phase1(
    models=MODELS,
    strategies=STRATEGIES,
    skip_existing=True,
)
print(f"\nWritten {len(written)} new result file(s).")

[skip] S1_multilingual-e5-large              recall@5_template_macro=0.601
[skip] S2-F1_multilingual-e5-large           recall@5_template_macro=0.685
[skip] S2-F2_multilingual-e5-large           recall@5_template_macro=0.712
[skip] S3-F1@0.25_multilingual-e5-large      recall@5_template_macro=0.706
[skip] S3-F1@0.5_multilingual-e5-large       recall@5_template_macro=0.689
[skip] S3-F1@0.75_multilingual-e5-large      recall@5_template_macro=0.637
[skip] S3-F2@0.25_multilingual-e5-large      recall@5_template_macro=0.694
[skip] S3-F2@0.5_multilingual-e5-large       recall@5_template_macro=0.702
[skip] S3-F2@0.75_multilingual-e5-large      recall@5_template_macro=0.669
[skip] S4_multilingual-e5-large              recall@5_template_macro=0.736
[skip] S1_bge-m3                             recall@5_template_macro=0.640
[skip] S2-F1_bge-m3                          recall@5_template_macro=0.698
[skip] S2-F2_bge-m3                          recall@5_template_macro=0.695
[skip] S3-F1@0.25_bge-m3 

In [3]:
# --- Phase 1 results: загрузка JSON, сводные таблицы, выбор (M*, S*) ---
def show_phase1_results():
    results_files = sorted(RESULTS_DIR.glob("*.json"))
    if not results_files:
        print("Нет результатов в experiments/results/. Запусти run_phase1() сначала.")
        return None

    rows = []
    for p in results_files:
        d = json.loads(p.read_text())
        if d.get("phase") != 1: continue
        pd_dict = d["metrics"].get("recall@5_template_per_dataset", {})
        rows.append({
            "exp_id": d["exp_id"],
            "strategy": d["config"]["strategy"],
            "model": d["config"]["model"].split("/")[-1],
            "macro":  d["metrics"]["recall@5_template_macro"],
            "min_ds": min(pd_dict.values()) if pd_dict else None,
            "films":  pd_dict.get("omhpbh1k83ao8"),
            "taxi":   pd_dict.get("33h8c3n5nbien"),
            "retail": pd_dict.get("b60rhj4luj0y3"),
            "observ": pd_dict.get("pzf0mu9kgqz4k"),
            "cross":  d["metrics"]["recall@5_template_cross_dataset"],
            "strict_macro": d["metrics"]["recall@5_strict_macro"],
        })

    import pandas as pd
    df = pd.DataFrame(rows)

    # 1) Pivot strategy × model по главной метрике
    print("== Pivot: recall@5_template_macro (rows = strategy, cols = model) ==\n")
    pivot = df.pivot(index="strategy", columns="model", values="macro").round(3)
    pivot = pivot.reindex(STRATEGIES)
    print(pivot.to_string())
    print()

    # 2) Per-model breakdown
    cols_show = ["strategy", "films", "taxi", "retail", "observ", "macro", "min_ds", "cross", "strict_macro"]
    for model in df["model"].unique():
        sub = df[df["model"] == model].copy()
        sub["_order"] = sub["strategy"].map({s: i for i, s in enumerate(STRATEGIES)})
        sub = sub.sort_values("_order")
        print(f"== {model} — все 10 стратегий ==\n")
        print(sub[cols_show].to_string(index=False, float_format='%.3f'))
        print()

    # 3) Top-10
    print("== TOP-10 over all 30 (macro desc, tie-break min_ds desc) ==\n")
    top = df.sort_values(["macro","min_ds"], ascending=[False,False]).head(10)
    print(top[["exp_id","macro","min_ds","films","taxi","retail","observ","cross","strict_macro"]]
          .to_string(index=False, float_format='%.3f'))
    print()

    winner = df.sort_values(["macro","min_ds"], ascending=[False,False]).iloc[0]
    weakest = winner[["films","taxi","retail","observ"]].astype(float).idxmin()
    print(f"→ Победитель (M*, S*): {winner['exp_id']}")
    print(f"   macro={winner['macro']:.3f}, min_ds={winner['min_ds']:.3f} ({weakest}), "
          f"cross={winner['cross']:.3f}, strict_macro={winner['strict_macro']:.3f}")
    return df

phase1_df = show_phase1_results()


== Pivot: recall@5_template_macro (rows = strategy, cols = model) ==

model       Qodo-Embed-1-1.5B  bge-m3  multilingual-e5-large
strategy                                                    
S1                      0.634   0.640                  0.601
S2-F1                   0.720   0.698                  0.685
S2-F2                   0.738   0.695                  0.712
S3-F1@0.25              0.723   0.687                  0.706
S3-F1@0.5               0.724   0.688                  0.689
S3-F1@0.75              0.699   0.650                  0.637
S3-F2@0.25              0.737   0.716                  0.694
S3-F2@0.5               0.734   0.719                  0.702
S3-F2@0.75              0.707   0.669                  0.669
S4                      0.742   0.698                  0.736

== Qodo-Embed-1-1.5B — все 10 стратегий ==

  strategy  films  taxi  retail  observ  macro  min_ds  cross  strict_macro
        S1  1.000 0.419   0.470   0.649  0.634   0.419  0.134         0.815
 

### Phase 1 — выводы

**Победитель:** `S4 + Qodo-Embed-1-1.5B`. macro `recall@5_template` = 0.742, min_ds = 0.500 (taxi), `recall@5_strict_macro` = 0.956 (Hit@5 source).

#### Главное наблюдение: схема компенсирует слабые описания

Per-dataset разница S1 (только description) → S4 (description + linearize_to_text) на Qodo:

| Датасет | S1 | S4 | Δ |
|---|---|---|---|
| films (хорошие осмысленные RU) | 1.000 | 1.000 | **+0.000** |
| observability (короткие EN) | 0.649 | 0.643 | -0.006 |
| taxi (содержательные RU) | 0.419 | 0.500 | +0.081 |
| **retail (демо: «Размер L», «таблица-2»)** | **0.470** | **0.823** | **+0.353** |

**Главный gain — на retail (+35%)**, где описания мусорные. На films (где описания и так идеальные) схема ничего не добавляет — S1 уже на 100%. Это и есть главная гипотеза проекта: **schema-aware embedding нужен ровно там, где описания не несут семантики**.

#### Top-strategies — спред маленький, выбор не критичен

Top-10 экспериментов: 0.716–0.742 (спред 0.026 — почти шум). Bootstrap CI на n=289 ≈ ±0.04 — разница между top-1 и top-10 в пределах статистической погрешности. **Выбор между S2-F2, S3-F2@*, S4 не принципиален.** Главный качественный скачок — S1 → schema-aware (+10–15% macro, +35% на retail).

#### Что узнали ещё

1. **F2 (NL-friendly) > F1 (JSON)** для всех моделей, включая qodo (контр-интуитивно — qodo задумана для кода, но на коротких рецептах NL-формат лучше).
2. **S4 (детерминированный NL-шаблон) — почти всегда лучший** для NL-моделей.
3. **Qodo-1.5B — самая сильная модель** при подключённой схеме. Без схемы (S1) она на уровне bge-m3.
4. **α-sweep в S3:** оптимум на α=0.25–0.5 (схема имеет вес 0.5–0.75). α=0.75 (description доминирует) хуже всегда.

#### Чем объясняется низкий recall на taxi (min_ds = 0.500)

**Это не проблема ретривера, а артефакт разметки.** Taxi имеет максимальные кластеры (29 рецептов, capped K=7), и в `relevant_template` для конкретного запроса попадают рецепты, *структурно* идентичные, но *тематически* далёкие.

Пример: запрос `«Динамика доли поездок в высоких тарифах в РФ»` (source = r_0007). В таргете три рецепта:
- r_0007 ✓ — про долю поездок в премиум-тарифах
- r_0011 — `«WoW Gmv в Москве»` (та же форма чарта, **другая тема**)
- r_0013 — `«доход водителей»` (та же форма, **другая тема**)

Retriever (правильно) ставит на топ рецепты, **тематически близкие** к запросу: r_0018 (corp trips Russia), r_0020 (intercity trips Russia), r_0015 (trip share by tariff Moscow). Но они **не помечены** template-релевантными. Recall@5 = 1/3 = 0.33 на этом запросе.

#### Реальное качество ретривала

`recall@5_strict_macro = 0.956` для победителя — это Hit@5 для source-recipe. То есть **ретривер находит правильный рецепт в top-5 в 95.6% случаев**. Метрика `recall@5_template = 0.742` ниже именно из-за structural-but-not-topical соседей в ground truth.

**Вывод:** для агента-чартостроителя реальное качество ближе к 0.96. Template-метрика штрафует за разумное поведение. Это надо учитывать в Phase 2/3 — стоит смотреть и template, и strict.

#### Решение по разметке

**Оставляем как есть.** Strict-метрика (0.956) уже отражает реальное качество. Template (0.742) — это «ceiling структурного охвата», валидный диагностический сигнал. Перерезать ground truth не нужно — достаточно при отчёте показывать обе и интерпретировать соответственно.

В Phase 2/3 будем считать обе и трактовать `strict` как primary signal качества.

#### Cross-dataset метрика

Низкая везде (0.04–0.14). T6 запросы требуют находить структурный шаблон в **другом домене** — dense-retrieval из коробки не справляется. Это задача для Phase 3 (BM25 / hybrid / reranker).


## Phase 2 — routing: (a) каскад vs (c) фиксированный mix

Сравниваются две стратегии fallback в сценарии C на победителе `(M*, S*)`:

**(a) Каскад с порогом τ:**
1. Считаем top-k по текущему датасету.
2. Если `max_score < τ` или `|results| < k_min` → расширяем поиск на весь корпус.
3. τ подбираем grid-search-ом, диапазон определяем по распределению score-ов после Phase 1.

**(c) Фиксированный mix:**
Всегда возвращаем top-k₁ из текущего датасета + top-k₂ из остальных. Параметры `k₁=3, k₂=2` по умолчанию (k=5 итого).

Главные метрики этапа: `routing_accuracy` (для (a)) + общий recall@5 на всём eval-сете.

## Phase 3 — лексические/гибридные baseline-ы на `(M*, S*)`

Цель — проверить, нужны ли в проде доп. компоненты или dense-only достаточно. По одному прогону на каждое:

- **BM25** на тексте лучшей стратегии (тот же `serialize`, что у dense-победителя).
- **Hybrid (BM25 + dense, RRF)** — fusion ранков.
- **Cross-encoder reranker** поверх dense top-20 → top-5.

## Phase 4 — fine-tuning (опционально, в конце)

Contrastive fine-tune лучшей dense-модели на train-сплите eval-сета.

**Split:** по `source_recipe_id` 70/15/15 (train/val/test). Test = финальный eval-сет, train/val для fine-tune. Без этого — утечка: модель «выучит» перефразировки тестовых рецептов.

**Триплеты:** `(query, positive=relevant_strict_recipe, negatives=hard_negatives + random)`.

## Что осознанно НЕ делаем

- **Query augmentation** (LLM-перефразировки рецептов на этапе индексации) — в проде LLM на стадии формирования рецептов недоступна.
- **`text-embedding-3-large`** — API-only, нет доступа.
- **nDCG / MRR** — downstream-агенту порядок не важен.

## Прогон Phase 2 — routing на победителе

Сравниваем (a) каскад с порогом τ vs (c) фиксированный mix top-k₁/k₂ на победителе Phase 1 = `S4 + Qodo-Embed-1-1.5B`.

**Что важно:** метрика Phase 2 отличается от Phase 1. В Phase 1 считали `recall@5` на `template ∩ current_dataset` с retrieval, ограниченным датасетом — это было «насколько хорошо ретривим в своём домене». В Phase 2 routing меняет, ОТКУДА брать кандидатов, поэтому метрика — это `recall@5` на **полном** `relevant_template` (включая cross-dataset). **Числа будут ниже**, потому что включают сложные cross-dataset шаблоны (films имеет 70% template'ов в других датасетах).

Логика — в `src/phase2.py` (`run_phase2()`, `show_phase2_results()`).

In [4]:
# --- Phase 2 runner: τ-sweep cascade + fixed mix + baseline ---
from src.phase2 import run_phase2
phase2_paths = run_phase2()
print(f"\nPhase 2: written {len(phase2_paths)} configs.")

[done] phase2_baseline_in_dataset_only                     recall@5_template_macro=0.408
[done] phase2_cascade_tau0.50                              recall@5_template_macro=0.408  routing_acc=0.859  fb_rate=0.06
[done] phase2_cascade_tau0.55                              recall@5_template_macro=0.405  routing_acc=0.862  fb_rate=0.08
[done] phase2_cascade_tau0.60                              recall@5_template_macro=0.404  routing_acc=0.841  fb_rate=0.13
[done] phase2_cascade_tau0.65                              recall@5_template_macro=0.407  routing_acc=0.790  fb_rate=0.21
[done] phase2_cascade_tau0.70                              recall@5_template_macro=0.411  routing_acc=0.737  fb_rate=0.34
[done] phase2_cascade_tau0.75                              recall@5_template_macro=0.412  routing_acc=0.611  fb_rate=0.51
[done] phase2_cascade_tau0.80                              recall@5_template_macro=0.413  routing_acc=0.425  fb_rate=0.76
[done] phase2_cascade_tau0.85                            

In [5]:
# --- Phase 2 results: сводная таблица по всем routing-конфигам ---
from src.phase2 import show_phase2_results
phase2_df = show_phase2_results()

== Phase 2 — все routing-конфиги (sorted by macro desc) ==

ВАЖНО: метрика отличается от Phase 1 — здесь recall@5 считается на ВСЁМ relevant_template
(включая шаблоны из других датасетов), а не на in_dataset-срезе.

                         exp_id          router   tau k1/k2  macro  films  taxi  retail  observ  strict_macro  routing_acc  fb_rate
         phase2_cascade_tau0.85         cascade 0.850     -  0.414  0.304 0.331   0.555   0.464         0.879        0.275    0.922
         phase2_cascade_tau0.80         cascade 0.800     -  0.413  0.300 0.331   0.555   0.464         0.879        0.425    0.760
           phase2_mix_k1_4_k2_1             mix     —   4/1  0.412  0.330 0.330   0.548   0.440         0.877            —        —
         phase2_cascade_tau0.75         cascade 0.750     -  0.412  0.294 0.335   0.554   0.464         0.869        0.611    0.509
         phase2_cascade_tau0.70         cascade 0.700     -  0.411  0.294 0.331   0.554   0.467         0.869        0.737  

### Phase 2 — выводы

**Лучшие routing-конфиги:** `cascade τ=0.85` (macro 0.414) и `mix k1=4/k2=1` (macro 0.412) — почти ровно. Baseline (только in-dataset, без fallback) = 0.408. **Прирост от routing — всего 0.4–0.6%.**

#### Где routing помогает, а где мешает

Per-dataset разница `cascade τ=0.85` vs `baseline (in_dataset_only)`:

| Датасет | baseline | τ=0.85 | Δ | Комментарий |
|---|---|---|---|---|
| films (70% cross-ds) | 0.280 | 0.304 | **+0.024** | fallback находит cross-dataset шаблоны |
| retail (mixed) | 0.546 | 0.555 | +0.009 | минимальный gain |
| taxi (65% in-ds) | 0.331 | 0.331 | 0 | в датасете и так всё, fallback не помогает |
| observ (74% in-ds) | 0.475 | 0.464 | **−0.011** | fallback **мешает**: в датасете много релевантного, выходить наружу = терять in-dataset слоты |

Маленькие датасеты (films) выигрывают от fallback, большие (observ) — теряют. Mix(4,1) находит компромисс: films получает один cross-dataset слот (films улучшается), но 4 in-dataset слота сохраняются для остальных датасетов.

#### Routing accuracy vs recall (cascade)

| τ | recall_macro | routing_acc | fallback_rate | Поведение |
|---|---|---|---|---|
| 0.50 | 0.408 | 0.859 | 6% | почти никогда не падбэк → как baseline |
| 0.65 | 0.407 | 0.790 | 21% | моделирует в основном правильно |
| 0.75 | 0.412 | 0.611 | 51% | переходный режим |
| 0.85 | 0.414 | 0.275 | 92% | почти всегда fallback → лучший recall, но routing_acc плох |

Видно: **высокий recall достигается за счёт «слишком частого» fallback** — routing accuracy при этом падает. Это говорит о том, что `fallback_label` ground truth (определение «правильно ли решил фоллбэчить») **не оптимизирован под recall** — оно консервативно отмечает большинство как `in_dataset`, тогда как реальный recall выигрывает от расширения поиска.

#### Главный вывод Phase 2

**Сценарий C (гибрид) — оверинженерия для нашего корпуса.** Cross-dataset queries — всего ~6% всего eval-set, и dense-retrieval даже с правильным fallback даёт ≤1.5% прироста на главной метрике. Простой `in_dataset_only` baseline или `mix(4,1)` работают практически так же, как сложный τ-cascade.

**Рекомендация для прода:** `mix(4,1)` — robust default без τ-настройки. На films он даёт лучший результат (0.330), на остальных сравнимо с baseline.

**Что важнее для Phase 3:** значение принципиально лучшего ретривала на in-dataset (taxi застрял на 0.331 при baseline) — это потолок dense-retrieval, который BM25/hybrid/reranker могут поднять.

#### Корректировка приоритетов Phase 3

Изначально планировалось проверить BM25 / hybrid / reranker на `(M*, S*)`. После Phase 2 видно: **главная задача Phase 3 — улучшить in-dataset ranking**, особенно на taxi. Cross-dataset (T6) можно отложить — гипотеза того, что hybrid там что-то добавит, не сильно подтверждается тем, что dense+fallback почти не помогает.

## Методологическая заметка: per-query-type breakdown + Held-out план

### Что показывает per-query-type разрез на победителе (S4 + Qodo)

| Тип запроса | n | recall_template_in_ds (Phase 1) | recall_template_full (Phase 2) | recall_strict (Hit@5 source) |
|---|---|---|---|---|
| T1 близкая перефразировка | 67 | 0.662 | 0.423 | **0.955** |
| T2 структурный намёк | 67 | 0.723 | 0.481 | **0.978** |
| T3 сущности/числа | 67 | 0.700 | 0.460 | **1.000** |
| T4 косвенная формулировка | 67 | 0.610 | 0.390 | **0.903** |
| T5 hard negative («близнец») | 19 | — | 0.318 | — |
| **T6 cross-dataset trigger** | **19** | — | **0.150** | **0.263** |
| T7 no-match | 28 | — | — | — |

**На лёгком случае (T1–T4) система работает блестяще** (Hit@5 = 90–100%). Если запрос — это перефраз существующего рецепта, находим его почти всегда.

**На тяжёлом случае (T6) система проваливается** (Hit@5 = 26%). Когда пользователь спрашивает про то, чего в его датасете нет, мы находим структурный шаблон из другого домена только в четверти случаев.

### Методологическая дыра

Распределение типов в нашем eval vs реалистичный продакшн:

| Сценарий | В eval | В продакшене (estimate) |
|---|---|---|
| Точный паттерн уже в датасете (T1–T4) | 80% | ~20–30% |
| Запрос «адаптируем чужой шаблон» (T5/T6) | 11% | ~50–60% |
| No-match (T7) | 8% | ~10–20% |

Наш headline `recall@5_template_macro = 0.742` (Phase 1) **сдвинут к лёгкому случаю**, а в проде доминирует тяжёлый. Это значит: **выбор `(M*, S*) = S4 + Qodo` оптимизирован под сценарий «вопрос ≈ описание из корпуса»**, который для нашего use case — меньшинство.

Что мы знаем дополнительно: на T6 **все стратегии и все модели плохие** (0.04–0.14 recall_template_cross). Schema-aware подход для T6 **не помогает** (делает эмбеддинг domain-specific, а T6 нужен domain-agnostic).

### Held-out эксперимент — план

**Идея.** Симулируем продакшен через leave-one-out: **поочерёдно удаляем каждый рецепт из корпуса** и смотрим, найдёт ли retrieval его структурный шаблон в оставшихся.

**Алгоритм** (для каждого пары `(модель, стратегия)`):

```
for r_X in recipes:                                  # 67 итераций
    corpus_held = recipes \ {r_X}                    # 66 рецептов
    for q in queries(source=r_X, type∈T1..T4):       # 4 запроса
        top5 = retrieve(q, corpus_held)              # ищем без r_X
        relevant_held = (relevant_template[q] - {r_X})  # все кроме r_X
        recall = |top5 ∩ relevant_held| / |relevant_held|
    aggregate per (current_dataset, query_type)
```

**Что измеряем:** `recall@5_held_out` — насколько хорошо находим **структурный шаблон**, когда «правильного» ответа в корпусе нет.

**Почему это правильная симуляция продакшена:**
- В реальности у юзера почти никогда нет точного match — есть «адаптировать существующий чарт».
- Held-out явно убирает «лёгкий» путь и заставляет ретривал работать в режиме «найти ближайшее».
- T1–T4 запросы переиспользуются как есть — они для рецепта r_X, но r_X удалён из корпуса. Получается production-like setup без генерации новых запросов.
- 67 рецептов × 4 query types = **268 production-like тестов** на каждую `(M, S)` пару (или 67 × 7 если включать T5/T6/T7). Достаточно для статистики.

**Ожидаемый результат:**
- recall@5_held_out на T1–T4 **сильно ниже** Phase 1 (0.74 → ?). Это уберёт «бонус» от наличия точного источника в корпусе.
- Если новый победитель отличается от Phase 1 → Phase 1 выбор был оптимизирован не под продакшн → пересматриваем.
- Если все стратегии одинаково плохи → подтверждается, что dense retrieval саму проблему «нет точного матча» решить не может, и **главная надежда на Phase 3** (BM25 / hybrid / cross-encoder rerank).

**Что добавится в `experiments/results/`:**
- `phase1_5_heldout_{model}_{strategy}.json` с метриками per query_type и per dataset
- Опционально: per-recipe breakdown (какие рецепты теряются хуже всего без помощи в корпусе — диагностика)


## Прогон Phase 1.5 — held-out leave-one-out (production-realistic выбор стратегии)

**Что меняется vs Phase 1.** В Phase 1 каждый запрос ищет в корпусе, **включая source-рецепт** (тот, из которого запрос был сгенерирован). В производстве source-рецепта не существует — пользователь просит чарт, которого в корпусе нет. Phase 1.5 симулирует это через **leave-one-out**: для каждого T1–T4 запроса убираем source из корпуса и проверяем, найдёт ли retrieval **другие структурно релевантные** рецепты.

**Конфигурация:**
- 10 стратегий × 3 модели = **30 прогонов** (Phase 1.5 — это **выбор стратегии**, не валидация одного победителя).
- Retrieval — `in_dataset_only` (как Phase 1), чтобы сравнение было apples-to-apples.
- Cap K=7 в `relevant_template_candidates.json` — используем существующий, фильтруем r_X (не пересчитываем).
- T1–T4 (268 запросов) — основная held-out метрика; T5/T6 (38) — sidecar на full retrieval без leave-one-out.
- T7 — игнорируем (precision-метрика, отдельный концерн).

**Стоимость.** Эмбеддинги корпуса и запросов уже закешированы Phase 1 — весь грид считается за секунды (только cosine-mat-ops).

Логика — в `src/phase1_5_heldout.py`. Структура схожа с Phase 1/2: `run_phase1_5()` пишет JSON в `experiments/results/phase1_5_*.json`, `show_phase1_5_results()` рендерит сводные таблицы (pivot, TOP-10, Phase 1 → 1.5 diff, per-query-type для победителя).

In [5]:
# --- Phase 1.5 runner: held-out leave-one-out на всех 30 (M, S) парах ---
# Логика в src/phase1_5_heldout.py. Idempotent: пропускает уже посчитанные.
# Эмбеддинги переиспользуются из кеша Phase 1 — весь грид прогоняется за секунды.

from src.phase1_5_heldout import run_phase1_5
phase1_5_paths = run_phase1_5(skip_existing=True)
print(f"\nPhase 1.5: written {len(phase1_5_paths)} new result(s).")

[skip] phase1_5_S1_multilingual-e5-large                   heldout_macro=0.542
[skip] phase1_5_S2-F1_multilingual-e5-large                heldout_macro=0.604
[skip] phase1_5_S2-F2_multilingual-e5-large                heldout_macro=0.613
[skip] phase1_5_S3-F1@0.25_multilingual-e5-large           heldout_macro=0.601
[skip] phase1_5_S3-F1@0.5_multilingual-e5-large            heldout_macro=0.585
[skip] phase1_5_S3-F1@0.75_multilingual-e5-large           heldout_macro=0.562
[skip] phase1_5_S3-F2@0.25_multilingual-e5-large           heldout_macro=0.582
[skip] phase1_5_S3-F2@0.5_multilingual-e5-large            heldout_macro=0.596
[skip] phase1_5_S3-F2@0.75_multilingual-e5-large           heldout_macro=0.574
[skip] phase1_5_S4_multilingual-e5-large                   heldout_macro=0.618
[skip] phase1_5_S1_bge-m3                                  heldout_macro=0.571
[skip] phase1_5_S2-F1_bge-m3                               heldout_macro=0.608
[skip] phase1_5_S2-F2_bge-m3                        

In [6]:
# --- Phase 1.5 results: held-out vs Phase 1 comparison, per-dataset + per-query-type ---
from src.phase1_5_heldout import show_phase1_5_results
phase1_5_df = show_phase1_5_results()

== Pivot Phase 1.5: held-out recall@5_template_macro (rows=strategy, cols=model) ==

model       Qodo-Embed-1-1.5B  bge-m3  multilingual-e5-large
strategy                                                    
S1                      0.536   0.571                  0.542
S2-F1                   0.606   0.608                  0.604
S2-F2                   0.628   0.599                  0.613
S3-F1@0.25              0.594   0.578                  0.601
S3-F1@0.5               0.586   0.579                  0.585
S3-F1@0.75              0.579   0.566                  0.562
S3-F2@0.25              0.638   0.609                  0.582
S3-F2@0.5               0.613   0.620                  0.596
S3-F2@0.75              0.585   0.570                  0.574
S4                      0.642   0.611                  0.618

== Qodo-Embed-1-1.5B — все 10 стратегий (Phase 1.5 held-out) ==

  strategy  films  taxi  retail  observ  ho_macro  min_ds    T1    T2    T3    T4    T5  T6_tmpl  T6_strict
        S

### Phase 1.5 — выводы

**Победитель Phase 1.5 = `S4 + Qodo-Embed-1-1.5B`** — тот же, что в Phase 1. macro held-out = **0.642** (vs Phase 1: 0.742, **Δ = -0.100**).

#### Главный вывод: победитель устойчив, но Phase 1 переоценивал систему на ~10 пунктов

| | Phase 1 | Phase 1.5 (held-out) | Δ |
|---|---|---|---|
| S4 + Qodo (победитель) | 0.742 | **0.642** | −0.100 |
| Среднее по всем 30 | 0.696 | **0.594** | −0.102 |
| S1 (только description) — среднее | 0.625 | **0.550** | −0.075 |

Phase 1 «надул» оценку на ~10 п. потому, что в 80% запросов source-recipe лежал в корпусе и dense его быстро находил. В production-realistic режиме (source убран) это преимущество исчезает.

**S1 «надулся» меньше всех** (−0.06 до −0.10) — без schema модель не «заучивает» конкретный source-recipe, ей нечего терять при его удалении. Schema-aware подходы стартуют выше, но и теряют больше при leave-one-out — в чистом виде остаётся тот же ≈10-пунктовый gap, что был в Phase 1.

#### Spread между стратегиями расширился, но не сильно

Top-10 Phase 1.5: **0.608 – 0.642 (спред 0.034)** vs Phase 1 top-10: 0.716 – 0.742 (спред 0.026). Чуть больше разрешения, но выбор между S2-F2, S3-F2@*, S4 на Qodo по-прежнему некритичен.

#### Per-dataset на победителе — taxi обвалился, films остался идеальным (из-за skip)

| Датасет | Phase 1 | Phase 1.5 | Δ | Комментарий |
|---|---|---|---|---|
| films (omhpbh1k83ao8) | 1.000 | **1.000** | 0 | Но 52/268 запросов **пропущены** (n_skipped) — у films мало in-dataset relevants (avg 1.77), часто после удаления source остаётся пусто. Когда есть другой in-ds match — находим всегда. |
| observ (pzf0mu9kgqz4k) | 0.643 | **0.659** | +0.016 | Робастен к leave-one-out (in-ds avg 4.46 — много structural соседей). |
| retail (b60rhj4luj0y3) | 0.823 | **0.669** | −0.154 | Заметное падение, но всё ещё высоко — schema-aware подход помогает даже без точного source. |
| **taxi (33h8c3n5nbien)** | **0.500** | **0.239** | **−0.261** | Главный обвал. Большие in-ds кластеры (avg 3.30 ≈ 3-4 structural matches), но dense ranks тематически-близкие выше → структурные релевантные не попадают в top-5. |

**Taxi — главное узкое место для Phase 3.** Cross-encoder rerank или hybrid BM25+dense должны помочь именно здесь.

#### Per-query-type на победителе

| Тип | recall@5 held-out | Комментарий |
|---|---|---|
| T1 близкая перефразировка | 0.475 | Перефраз description — без source модель путается, описаний у других в датасете часто разные. |
| T2 структурный намёк | **0.551** | **Лучший** — намёк про тип чарта прямо ложится в schema-сериализацию S4. |
| T3 сущности/числа | 0.509 | Значения из filters частично попадают в S4. |
| T4 косвенная формулировка | **0.395** | **Худший** — нет ни description-сигналов, ни schema-сигналов. |

#### Sidecar T5/T6 — подтверждают тяжесть «нет точного матча»

| | recall@5 |
|---|---|
| T5 (hard negative «близнец»), full retrieval | 0.318 |
| T6 (cross-dataset trigger), template, full retrieval | 0.150 |
| T6, strict (точный cross-dataset twin), full retrieval | 0.263 |

T6 — это production scenario в чистом виде (запрос про то, чего в датасете пользователя нет, нужно найти структурный шаблон в другом домене). Dense из коробки даёт 15% — **это потолок для Phase 3**: BM25/hybrid/reranker должны атаковать именно эту дыру.

#### Что осознали

1. **Production-realistic headline = 0.64** (а не 0.74). Для отчёта/рекомендации продакшну использовать именно эту цифру.
2. **Качественный выбор (M*, S*) не меняется** — `S4 + Qodo-Embed-1-1.5B` остаётся рекомендацией.
3. **Главная цель Phase 3 — taxi (0.24) + T6 (0.15)**. На retail/observ dense уже хорош, films шумит из-за малой выборки. Boost именно в этих двух дырах.
4. **Schema-aware подход подтверждён**: S1 (только description) в среднем 0.550, schema-aware top — 0.642. +9 пунктов — стабильный выигрыш и в production-realistic режиме.

## Прогон Phase 3 — BM25 + Hybrid (RRF) поверх Phase 1.5 winner

**Цель.** Атаковать конкретные дыры Phase 1.5 на `S4 + Qodo`:
- **taxi in-dataset** (held-out macro = 0.239) — dense ставит тематически-близкие выше структурно-близких.
- **T6 cross-dataset template** (recall@5 = 0.150) — dense из коробки не работает на cross-domain структурном поиске.

**Конфигурация:**
- **Эксперимент 1 — BM25 alone.** Индекс на тексте S4 (`description + linearize_to_text(recipe)`), regex-токенизация (lowercase, Unicode `\w+`).
- **Эксперимент 2 — Hybrid (BM25 + dense, RRF).** Top-20 от BM25 + top-20 от Qodo+S4 → Reciprocal Rank Fusion (`1/(60+rank)`) → top-5.
- **Cross-encoder reranker** — осознанно вне scope этого прохода. Если придётся возвращаться к T1/T4 — следующий шаг.

**Eval — тот же, что и Phase 1.5** (production-realistic):
- T1–T4: in_dataset_only retrieval с leave-one-out source-рецепта.
- T5/T6: sidecar на full corpus retrieval, без leave-one-out.

Логика — в `src/phase3.py`. Зависимость — `rank_bm25` (`pip install rank_bm25`).

In [7]:
# --- Phase 3 runner: BM25 + Hybrid (RRF) ---
# Логика в src/phase3.py. Eval-сетап — Phase 1.5 (held-out leave-one-out на T1-T4, T5/T6 sidecar).
# Idempotent: пропускает уже посчитанные. BM25 строится за миллисекунды, dense эмбеддинги — из кеша.

from src.phase3 import run_phase3
phase3_paths = run_phase3(skip_existing=True)
print(f"\nPhase 3: written {len(phase3_paths)} new result(s).")

[skip] phase3_bm25_S4                            heldout_macro=0.639
[skip] phase3_hybrid_rrf                         heldout_macro=0.640

Phase 3: written 0 new result(s).


In [8]:
# --- Phase 3 results: BM25, Hybrid vs Phase 1.5 dense baseline ---
from src.phase3 import show_phase3_results
phase3_df = show_phase3_results()

== Phase 3 vs Phase 1.5 baseline (held-out, T1–T4 main; T5/T6 sidecar) ==

            approach  macro  min_ds  films  taxi  retail  observ
dense S4+Qodo (P1.5)  0.642   0.239  1.000 0.239   0.669   0.659
             bm25_S4  0.639   0.254  1.000 0.254   0.653   0.648
          hybrid_rrf  0.640   0.228  1.000 0.228   0.653   0.678

== Per query-type (held-out T1–T4) + sidecar T5/T6 ==

            approach    T1    T2    T3    T4    T5  T6_tmpl  T6_strict
dense S4+Qodo (P1.5) 0.475 0.551 0.509 0.395 0.318    0.150      0.263
             bm25_S4 0.468 0.532 0.522 0.417 0.200    0.254      0.421
          hybrid_rrf 0.498 0.523 0.500 0.389 0.230    0.237      0.474

== Δ vs dense S4+Qodo baseline (positive = approach лучше) ==

  approach  Δ_macro  Δ_taxi  Δ_retail  Δ_observ  Δ_T6_tmpl  Δ_T6_strict
   bm25_S4   -0.003  +0.015    -0.015    -0.011     +0.105       +0.158
hybrid_rrf   -0.002  -0.012    -0.015    +0.019     +0.087       +0.211


### Phase 3 — выводы

**Цель — атаковать две дыры Phase 1.5:** taxi in-dataset (0.239) и T6 cross-dataset template (0.150). Реализация — BM25 + Hybrid (RRF) поверх dense-победителя `S4 + Qodo`.

#### На headline-метрике (in-dataset T1–T4) — паритет

| Подход | held-out macro | min_ds (taxi) | films | retail | observ |
|---|---|---|---|---|---|
| dense S4+Qodo (Phase 1.5 baseline) | **0.642** | 0.239 | 1.000 | 0.669 | 0.659 |
| BM25 alone | 0.639 | **0.254** | 1.000 | 0.653 | 0.648 |
| Hybrid (BM25 + dense, RRF) | 0.640 | 0.228 | 1.000 | 0.653 | **0.678** |

**Δ macro vs baseline: −0.003 (BM25), −0.002 (Hybrid)** — все три ≈ одинаково. Главный сюрприз: **BM25 alone уже на уровне с самым дорогим dense pipeline'ом**. Schema-aware S4-текст работает для BM25 через лексический матч (тип чарта, названия полей).

#### Где BM25/Hybrid реально побеждают: T6 cross-dataset

| | dense (P1.5) | BM25 | Hybrid | Δ vs dense |
|---|---|---|---|---|
| **T6_template** | 0.150 | **0.254** | 0.237 | BM25: +0.105, Hybrid: +0.087 |
| **T6_strict** | 0.263 | 0.421 | **0.474** | BM25: +0.158, Hybrid: **+0.211** |

**T6_strict у Hybrid = 0.474** — почти **2× к dense baseline'у**. Это и есть главное production-realistic улучшение: когда пользователь спрашивает про чарт, которого в его датасете нет, **Hybrid в 47% случаев находит правильный cross-dataset twin в top-5** (vs 26% у чистого dense).

**Почему BM25/Hybrid лучше на cross-dataset:** dense-эмбеддинг тянет к тематически-близким (одно слово/контекст), а cross-dataset требует структурной близости (одинаковый тип чарта + роли полей, но в чужом домене). BM25 чаще ловит лексический матч на schema-словах (`«флэт-таблица»`, `«столбчатый»`, `«долю»`), которые присутствуют в S4-сериализации.

#### На taxi (in-dataset, главная dense-дыра) — небольшой gain от BM25

| | dense | BM25 | Hybrid |
|---|---|---|---|
| taxi held-out | 0.239 | **0.254** | 0.228 |

BM25 даёт +1.5 п. на taxi. Hybrid здесь слабее (−1.1 п.) — лексика BM25 «дробит» dense ranking, и dense теряет хороший top-5 ранк. Так бывает: для in-dataset запросов dense ranks tematically близкие правильно, а BM25 портит верх top-5 структурно-близкими-но-семантически-далёкими.

#### Per query-type на T1–T4

| Тип | dense | BM25 | Hybrid | Что узнали |
|---|---|---|---|---|
| T1 близкая перефразировка | 0.475 | 0.468 | **0.498** | Hybrid усиливает dense на близкой перефразировке. |
| T2 структурный намёк | **0.551** | 0.532 | 0.523 | Dense лучший — S4-эмбеддинг ловит «линией»/«столбиками». |
| T3 сущности/числа | 0.509 | **0.522** | 0.500 | BM25 лучший — точные значения из filters сильнее в лексике. |
| T4 косвенная формулировка | 0.395 | **0.417** | 0.389 | BM25 неожиданно лучше — но T4 худший всегда. |

#### Рекомендация для прода

**Hybrid (BM25 + dense S4+Qodo, RRF k=60).** Аргументы:
1. На in-dataset — паритет с dense (потеря ≤0.002).
2. **На cross-dataset T6 — major win**: T6_strict 0.474 vs dense 0.263 — почти 2×.
3. Не требует дополнительных моделей (BM25 пренебрежимо лёгкий, dense эмбеддинги уже считаем).
4. Простая реализация и явная семантика (RRF — стандарт).

**Что делать с taxi (0.228 у Hybrid, 0.254 у BM25).** Соблазнительно сделать per-dataset routing (taxi → BM25, остальные → Hybrid) — даст +0.008 macro. **Но эта техника не масштабируется**: каждый новый датасет требует ручного A/B и hardcoded if/else. В production retrieval pipeline решение всегда должно быть **универсальным** — один retriever на все датасеты. Правильный путь для taxi-дыры — universal layer поверх Hybrid (cross-encoder reranker) или улучшение сериализации (field-role в S4), а не датасет-специфичная развилка.

**Что осталось непроверенным:** cross-encoder reranker. По плану отложили; гипотетически мог бы поднять T1/T4 на in-dataset и retail/T5. Если придётся возвращаться — следующий шаг именно это.

## Что делать дальше (когда вернёшься к проекту)

**Состояние:** Проект завершён. Phase 1 + Phase 2 + Phase 1.5 + Phase 3 + итоговый отчёт (Шаг 10). Артефакты разметки в `data/eval/`, результаты в `experiments/results/` (30 Phase 1 + 12 Phase 2 + 30 Phase 1.5 + 2 Phase 3). Все ячейки оркестрации работают idempotent. Финальный отчёт и production-рекомендация — в последней ячейке ноутбука («## Итоговый отчёт»).

> **Production-realistic headline:** `recall@5_template_macro = 0.642` (Phase 1.5, held-out leave-one-out на победителе `S4 + Qodo`). Phase 1 цифра 0.742 — на «лёгком» случае (source-recipe в корпусе). Для отчёта и сравнений с Phase 3 использовать Phase 1.5 числа.

> **Принцип ноутбука:** все эксперименты (Phase 1–4) **запускаются и отображаются прямо отсюда**. Логика — в `src/`, ноутбук импортирует и вызывает.

1. ✅ **Шаг 1: Препроцессинг ролей полей** — `data/field_roles.json`. 108 полей.

2. ✅ **Шаг 2: Generator-pass** — `data/eval/queries_raw.jsonl`, 334 запроса.

3. ✅ **Шаг 3: Judge-pass** — `data/eval/queries.jsonl` с обеими метками + `fallback_label`.

4. ✅ **Шаг 4: Ручной ревью** — пройден.

5. ✅ **Шаг 5: Phase 1 — основной грид (30 прогонов).** Победитель `(M*, S*)` = `S4 + Qodo-Embed-1-1.5B`. macro `recall@5_template` = 0.742, `recall@5_strict` = 0.956.

6. ✅ **Шаг 6: Phase 2 — routing.** Лучший роутер ≈ `mix(4,1)` или `cascade τ=0.85` (macro 0.412–0.414). Прирост над baseline ~1%.

7. ✅ **Шаг 7: Phase 1.5 — held-out leave-one-out (production-realistic выбор стратегии).** Все 30 пар (M, S) прогнаны с leave-one-out source-recipe.

   **Результат:** победитель остался прежним — `S4 + Qodo-Embed-1-1.5B`. macro held-out = **0.642** (vs Phase 1: 0.742, Δ = −0.100). Phase 1 переоценивал систему на ~10 пунктов: source-recipe лежал в корпусе в 80% запросов, и dense его быстро находил. На production-realistic режиме это преимущество исчезает.

   **Что узнали:**
   - Качественный выбор `(M*, S*)` устойчив — schema-aware на Qodo лучшая пара в обоих режимах.
   - Spread top-10 = 0.034 (Phase 1: 0.026) — чуть шире, но выбор внутри top-стратегий по-прежнему некритичен.
   - Главные дыры: **taxi (0.239)** и **T6 cross-dataset (0.150)**. На retail/observ dense уже работает хорошо.
   - **Headline для отчёта = 0.642**, а не 0.742.

8. ✅ **Шаг 8: Phase 3 — BM25 + Hybrid (RRF) на победителе.** Eval-сетап — Phase 1.5 (held-out leave-one-out).

   **Результаты:**
   - **In-dataset (T1–T4 held-out):** все три подхода паритет — dense 0.642 / BM25 0.639 / Hybrid 0.640. BM25 alone уже на уровне с dense.
   - **T6 cross-dataset (главное улучшение):** T6_strict 0.263 (dense) → 0.421 (BM25) → **0.474 (Hybrid)**. Почти 2× boost — главное production-realistic улучшение.
   - **Taxi (in-dataset дыра):** BM25 alone 0.254, Hybrid 0.228, dense 0.239. BM25 даёт +1.5 п., Hybrid −1.1 п. Taxi-плато не пробивается одним rerank-step.
   - Cross-encoder reranker осознанно вне scope (мог бы поднять T1/T4 — следующий step если придётся).

   **Рекомендация для прода:** `Hybrid (BM25 + dense Qodo+S4, RRF k=60)`. На in-dataset паритет, на cross-dataset major win, без дополнительных моделей.

9. ⬜ **Шаг 9: Phase 4 — fine-tuning** (опционально). Contrastive на лучшей модели Qodo-1.5B, split по `source_recipe_id` 70/15/15. Скорее всего, малый размер eval'а (n=334) сделает дообучение нестабильным — это «опциональная» цель.

10. ✅ **Шаг 10: Итоговый отчёт** — финальная markdown-ячейка в конце ноутбука («## Итоговый отчёт»). Содержит:
    - TL;DR + production-рекомендация (Hybrid BM25+dense, RRF k=60)
    - Сводная таблица всех Phase (1, 1.5, 2, 3) с headline-метриками
    - Проверка главной гипотезы проекта (schema-aware компенсирует слабые описания)
    - Что узнали про каждый датасет, taxi-аномалия как metric artifact
    - Что не получилось (taxi 0.24, T4, cross-encoder, eval-set bias)
    - Anti-pattern'ы для прода (per-dataset routing — не масштабируется)
    - Путь к 0.80 — стек универсальных улучшений с ожидаемыми вкладами

**Проект завершён.** Все основные шаги (1–8, 10) выполнены. Шаг 9 (Phase 4 fine-tuning) оставлен как опциональный — не рекомендуется на текущем размере eval-сета (334 queries, риск overfit'а высокий).

## Структура проекта

```
project/
├── data/
│   ├── recipes.json                    # ✅ корпус (67 рецептов)
│   ├── fields_by_dataset.json          # ✅ список полей по датасетам
│   ├── field_roles.json                # ✅ роли полей (Шаг 1)
│   ├── recipe_template_candidates.json # ✅ компат-списки top-7 (для Judge-pass)
│   └── eval/
│       ├── _generate_t1_t4.py          # ✅ генератор T1–T4 запросов
│       ├── _generate_t5_t6_t7.py       # ✅ генератор T5/T6/T7 запросов
│       ├── _compute_signatures.py      # ✅ fuzzy compat + similarity cap
│       ├── _judge.py                   # ✅ judge с обеими метками
│       ├── queries_raw.jsonl           # ✅ после Generator-pass
│       └── queries.jsonl               # ✅ финальный eval после Judge-pass
├── src/
│   ├── config.py                       # ✅ грид моделей/стратегий, пути
│   ├── indexers/
│   │   ├── serialize.py                # ✅ flat_kv(), linearize_to_text(), json_dump()
│   │   └── embed.py                    # ✅ обёртки над e5/BGE/qodo + кеш
│   ├── retrievers/
│   │   └── dense.py                    # ✅ cosine top-k
│   ├── eval/
│   │   └── metrics.py                  # ✅ recall@k, macro/micro/per-dataset
│   ├── phase1.py                       # ✅ runner Phase 1
│   ├── phase2.py                       # ✅ runner Phase 2 (cascade + mix)
│   ├── phase1_5_heldout.py             # ✅ leave-one-out runner (Шаг 7)
│   └── phase3.py                       # ✅ BM25 + Hybrid RRF (Шаг 8)
├── experiments/
│   ├── cache/                          # ✅ эмбеддинги по (model, text_hash)
│   └── results/
│       ├── phase1: {S*}_{model}.json   # ✅ 30 файлов
│       ├── phase1_5_*.json             # ✅ 30 файлов
│       ├── phase2_*.json               # ✅ 12 файлов
│       └── phase3_*.json               # ✅ 2 файла (BM25, Hybrid RRF)
├── chartRecipes.json                   # оригинал, не модифицируется
└── project.ipynb                       # этот блокнот: оркестрация, прогоны, отчёт
```

## Формат записи результата эксперимента (Phase 1)

```json
{
  "exp_id": "S2-F2_bge-m3",
  "phase": 1,
  "config": {"strategy": "S2-F2", "model": "BAAI/bge-m3", "alpha": null, "format": "F2", "k_top": 10},
  "metrics": {
    "recall@5_template_macro": 0.71, "recall@5_template_micro": 0.74,
    "recall@5_template_per_dataset": {"omhpbh1k83ao8": 0.86, ...},
    "recall@5_template_cross_dataset": 0.61,
    "recall@5_strict_macro": 0.85,
    "recall@1_template_macro": 0.52, "recall@10_template_macro": 0.89,
    "latency_ms_p50": 14
  },
  "timestamp": "2026-05-10T16:00:00",
  "corpus_size": 67, "eval_size": 334
}
```

## Формат записи Phase 2

```json
{
  "exp_id": "phase2_cascade_tau0.70",
  "phase": 2,
  "config": {"router": "cascade", "tau": 0.70, "k_min": 3, "k_final": 5,
             "model": "Qodo/Qodo-Embed-1-1.5B", "strategy": "S4"},
  "metrics": {
    "recall@5_template_macro": 0.411,
    "recall@5_template_per_dataset": {...},
    "recall@5_strict_macro": 0.869,
    "routing_accuracy": 0.737,
    "fallback_rate": 0.335
  },
  ...
}
```


## Итоговый отчёт

### TL;DR

**Production-рекомендация:** `Hybrid (BM25 + dense Qodo+S4, RRF k=60)`. Универсальный pipeline для всех датасетов, без датасет-специфичных настроек.

**Production-realistic метрики** (Phase 1.5 held-out + Phase 3, leave-one-out source-рецепта):

| Метрика | Значение | Что значит |
|---|---|---|
| recall@5_template_macro (in-dataset T1–T4) | **0.640** | в среднем находим 64% структурно-релевантных рецептов в top-5, когда точного матча в корпусе нет |
| recall@5_strict (T6 cross-dataset) | **0.474** | в 47% случаев находим правильный cross-dataset twin в top-5 (vs 26% у чистого dense) |

### Сводная таблица всех Phase

Headline на одной картине (победитель `S4 + Qodo-Embed-1-1.5B`, eval — production-realistic held-out с leave-one-out):

| Phase | Описание | macro held-out | T6_template | T6_strict |
|---|---|---|---|---|
| Phase 1 (диагностика) | dense, source-recipe в корпусе | 0.742 | 0.073 | 0.956 |
| Phase 1.5 (production-realistic) | dense S4+Qodo, source убран | 0.642 | 0.150 | 0.263 |
| Phase 2 | routing на dense | ~ 0.640 | — | — |
| Phase 3 — BM25 | лексика на S4-тексте | 0.639 | 0.254 | 0.421 |
| **Phase 3 — Hybrid (recommended)** | **BM25 + dense, RRF k=60** | **0.640** | 0.237 | **0.474** |

**Главное:** между Phase 1 (0.742) и Phase 1.5 (0.642) разница 10 п. — Phase 1 переоценивал систему потому, что source-рецепт лежал в корпусе. Production-realistic headline — 0.640, а не 0.742.

**Phase 3 Hybrid даёт паритет с dense на in-dataset, но почти 2× на cross-dataset T6_strict** (0.474 vs 0.263). На каждом 2-м запросе про «чего нет в моём датасете» система теперь находит правильный шаблон из другого домена — это и есть основное production-realistic улучшение.

### Главная гипотеза проекта и её проверка

**Гипотеза:** schema-aware embedding должен компенсировать слабые описания (retail имеет описания вида `«Размер L»`, `«таблица-2»` — semantically бесполезны).

**Подтверждена:** на retail dense + S4-сериализация даёт **0.669** vs S1 (только description) **0.292** в held-out. **+38 пунктов**, разница огромная.

**Расширение гипотезы:** schema-сериализация важна для всех retriever'ов, не только dense. BM25 на S4-тексте даёт **0.653** на retail. То есть **главный сигнал — это сам факт schema-сериализации**, а не выбор retriever-а.

### Что узнали про данные

| Датасет | n | Особенность описаний | Held-out macro |
|---|---|---|---|
| films (omhpbh1k83ao8) | 7 | длинные осмысленные RU | 1.000* |
| taxi (33h8c3n5nbien) | 29 | RU + эмодзи, большие structural кластеры | 0.239 |
| retail (b60rhj4luj0y3) | 19 | мусорные (`«Размер L»`, `«таблица-2»`) | 0.669 |
| observ (pzf0mu9kgqz4k) | 12 | короткие EN slug | 0.659 |

*films шумит — 52/268 запросов skip из-за малой выборки in-dataset relevants.

**Taxi-аномалия:** 0.239 — самая низкая цифра, но это **в значительной степени metric artifact**. Taxi имеет большие structural кластеры (29 рецептов → cap K=7 в `relevant_template`); retrieval корректно ставит тематически-близкие в top-5, но они **не помечены** template-релевантными (только структурно-эквивалентные). Подтверждение: `recall@5_strict` (узкая метрика «нашли ли gold source») на taxi >0.9 в Phase 1.

### Что не получилось

1. **Taxi in-dataset (0.24) не пробивается** dense / BM25 / Hybrid. Скорее всего — metric artifact (см. выше), но окончательно не подтверждено без ручного аудита.
2. **T4 (косвенная формулировка) — самый слабый везде**, 0.39–0.42. Без description-сигналов и schema-намёков retrieval барахтается. Нужен query understanding или multi-query rewriting.
3. **Cross-encoder reranker не реализован** — отложен по плану. Гипотетически закрыл бы T1/T4 и taxi (+5–10 п. macro).
4. **Eval-set bias:** 334 LLM-generated queries по шаблонам T1–T7. На реальных пользовательских запросах система не тестировалась — это самый слабый методологический момент проекта.

### Чего НЕ нужно делать в production

- **Per-dataset routing** (`if dataset_id == "taxi": ...`) — даёт +0.008 macro, но **не масштабируется**: каждый новый датасет требует ручного A/B и обновления конфига. Production retrieval pipeline должен быть **универсальным** — один подход на все домены, либо универсальный layer сверху (cross-encoder reranker).
- **Fine-tuning на 334 queries** — sample too small, риск overfit'а высокий.
- **Бóльшая dense-модель** (Qodo-7B и выше) — diminishing returns, инфраструктурная нагрузка непропорциональна.

### Путь к более высоким цифрам (если потребуется)

Если 0.640 недостаточно и нужно к 0.80 — реалистичный стек улучшений (по убыванию ожидаемого вклада):

| Действие | Ожидаемый вклад | Универсально? |
|---|---|---|
| Cross-encoder reranker (bge-reranker-v2-m3) поверх Hybrid top-20 | +0.05–0.08 | да |
| Аудит taxi-eval руками (вероятно 50% низкой цифры — metric artifact) | +0.03–0.05 | да (eval correction) |
| Field-role явно в S4: `«поле "Год выпуска" (temporal)»` | +0.02–0.03 | да |
| Pymorphy3 lemmatization для BM25 | +0.01–0.03 | да |
| Multi-query retrieval (LLM rewrite + RRF) | +0.02–0.03 | да |
| Переход на `recall@10` как primary metric (если downstream handles 10 seeds) | +0.05–0.08 | да (методологически) |

Все улучшения **универсальные** — работают на любом новом датасете без перенастройки. Стек оптимистично даёт ~0.80, реалистично 0.72–0.78.

**Принципиальный момент:** правильная следующая инвестиция — это **не +N процентов на этом eval'е**, а **расширение eval'а до 1500+ real-user queries**. Текущий 334 LLM-generated set с известным bias не даст представительных production-цифр, сколько ни тюнинговать retriever. Качественная валидация на 20–30 настоящих пользовательских запросов даст больше пользы, чем cross-encoder reranker для красивого числа.

### Финальная рекомендация

Деплоим **Hybrid (BM25 + dense Qodo+S4, RRF k=60)** как universal retrieval pipeline. Перед раскаткой — собрать ~30 реальных юзер-запросов и провалидировать качество top-5 руками. Если retrieval приемлемый — катить, мониторить, итерировать на реальных данных. Все academic-tuning'и (cross-encoder, multi-query) — на основе production-signal, а не на синтетическом eval.